<a href="https://colab.research.google.com/github/sevval-345/SoftITo/blob/main/temel_ann.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 Temel Yapay Sinir Ağı (ANN)
PyTorch ile ikili sınıflandırma problemi.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

## 1. Veri Hazırlama

In [2]:
X, y = make_classification(n_samples=1000, n_features=10, n_classes=2, random_state=42)
X = StandardScaler().fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# NumPy → Tensor
X_train = torch.FloatTensor(X_train)
X_test  = torch.FloatTensor(X_test)
y_train = torch.FloatTensor(y_train).unsqueeze(1)  # (800,) → (800, 1)
y_test  = torch.FloatTensor(y_test).unsqueeze(1)

print(f"Eğitim: {X_train.shape} | Test: {X_test.shape}")

Eğitim: torch.Size([800, 10]) | Test: torch.Size([200, 10])


## 2. Model Tanımı


In [3]:
class ANN(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(10, 32),   # Girdi: 10 özellik → 32 nöron
            nn.ReLU(),
            nn.Linear(32, 16),   # Gizli katman
            nn.ReLU(),
            nn.Linear(16, 1),    # Çıktı: tek nöron
            nn.Sigmoid()         # [0, 1] aralığına sıkıştır
        )

    def forward(self, x):
        return self.model(x)

model     = ANN()
kayip_fn  = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(model)
print(f"Parametre sayısı: {sum(p.numel() for p in model.parameters()):,}")

ANN(
  (model): Sequential(
    (0): Linear(in_features=10, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=16, bias=True)
    (3): ReLU()
    (4): Linear(in_features=16, out_features=1, bias=True)
    (5): Sigmoid()
  )
)
Parametre sayısı: 897


## 3. Eğitim




In [4]:

for epoch in range(1, 101):
    model.train()
    optimizer.zero_grad()               # Gradyanları sıfırla
    tahmin = model(X_train)             # İleri yayılım
    kayip  = kayip_fn(tahmin, y_train)  # Kayıp hesapla
    kayip.backward()                    # Geri yayılım
    optimizer.step()                    # Ağırlıkları güncelle

    if epoch % 20 == 0:
        model.eval()
        with torch.no_grad():
            test_pred = model(X_test)
            test_acc  = ((test_pred > 0.5).float() == y_test).float().mean()
        print(f"Epoch {epoch:3d} | Kayıp: {kayip.item():.4f} | Test Acc: {test_acc*100:.1f}%")

Epoch  20 | Kayıp: 0.6681 | Test Acc: 57.0%
Epoch  40 | Kayıp: 0.6058 | Test Acc: 78.5%
Epoch  60 | Kayıp: 0.5119 | Test Acc: 81.5%
Epoch  80 | Kayıp: 0.4138 | Test Acc: 83.5%
Epoch 100 | Kayıp: 0.3459 | Test Acc: 84.5%


## 4. Sonuç

In [5]:
model.eval()
with torch.no_grad():
    final = model(X_test)
    acc   = ((final > 0.5).float() == y_test).float().mean()
print(f"Final Test Doğruluğu: {acc*100:.2f}%")

Final Test Doğruluğu: 84.50%
